# Module 5 Lab: Improving an Agent with Evals

This notebook demonstrates the complete **eval-iterate cycle** for the Northwind customer support agent. Rather than treating evaluation as a final quality check, we use evaluation results to identify concrete failures, improve the agent, measure the impact of those improvements, and continuously expand our test coverage.

The workflow is organized into six primary stages:

* **Setup and Baseline:** Configure the evaluation environment and reproduce a concrete failure in the baseline `v1` agent, where unsupported product details are presented to the customer.
* **Build the Evaluations:** Define evaluators and create targeted datasets containing both known failure cases and known-good behavior for regression testing.
* **Improve the Agent:** Develop improved prompt versions (`v2` and `v3`) and run experiments across the evaluation datasets.
* **Compare the Experiments:** Analyze how each prompt version affects response grounding while also measuring tradeoffs such as tool calls and latency.
* **Evaluate Production Traffic:** Apply the same evaluation approach to production traffic to identify failures and edge cases that may not be represented in the offline datasets.
* **Close the Loop:** Promote meaningful production failures into permanent offline test cases, allowing future iterations to be evaluated against real-world behavior.

The goal is to turn evaluation into a continuous improvement process: identify failures, measure them, improve the agent, verify the improvement, and feed new production failures back into the evaluation suite.


## Table of Contents

- [Imports](#Imports)
- [1 - Setup and Baseline](#1---setup-and-baseline)
- [2 - Build the Evaluations](#2---build-the-evaluations)
    - **[Exercise 1 - policy_and_grounding_classifiers](#exercise-1--policy_and_grounding_classifiers)** 
- [3 - Improve the Agent](#3---improve-the-agent)
    - **[Exercise 2 - run_experiment](#exercise-2--run_experiment)** 
- [4 - Compare the Experiments](#4---compare-the-experiments)
- [5 - Evaluate Production Traffic](#5---evaluate-production-traffic)
    - **[Exercise 3 - production_traffic](#exercise-3--production_traffic)** 
- [6 - Close the Evaluation Loop](#6---close-the-evaluation-loop)

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, make sure you ran all of the unit tests and then click on the `Submit assignment` button on the top right of the page.
---

**Set your API keys**

The agent needs credentials for two services: 
- Anthropic, to call the model
- Phoenix, to receive the traces.

You need to set these up as environment variables for the SDKs to work correctly.

You also need to define a collector endpoint which is the address traces get sent to. Without it, the agent still runs but nothing lands in Phoenix.

Within the DLAI platform these are already set up for you but if you are working outside it make sure to define three variables yourself:

| Variable | Where to get it |
|---|---|
| `ANTHROPIC_API_KEY` | [console.anthropic.com](https://console.anthropic.com/settings/keys) → Settings → API Keys |
| `PHOENIX_API_KEY` | [app.phoenix.arize.com](https://app.phoenix.arize.com/login) → your Space → Settings |
| `PHOENIX_COLLECTOR_ENDPOINT` | Same Settings page, listed as your Hostname |

You can do this by creating a cell and placing this code along with the correct information:
```python
import os

os.environ["ANTHROPIC_API_KEY"] = "your-actual-api-key-here"
os.environ["PHOENIX_API_KEY"] = "your-actual-api-key-here"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "your-actual-endpoint-here"
```

<a id="imports"></a>
## Imports

The notebook begins by importing the libraries and utilities required for tracing, experimentation, evaluation, and interaction with the Northwind agent. 

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json

import phoenix as px
from phoenix.client import AsyncClient
from phoenix.client.experiments import async_run_experiment, create_evaluator
from phoenix.evals import LLM, ClassificationEvaluator
from phoenix.trace import using_project

import utils
import unittests
import pandas as pd

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: dlai-ecommerce-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



The Phoenix application and clients are initialized next. These clients provide access to tracing, datasets, annotations, and asynchronous experiment execution throughout the evaluation workflow.

In [2]:
OFFLINE_PROJECT = "dlai-ecommerce-agent"
PRODUCTION_PROJECT = "dlai-northwind-production"
JUDGE_LLM = LLM(provider="anthropic", model="claude-sonnet-4-6")

In [3]:
# Launch Phoenix
px.launch_app()

# Show the true Phoenix URL
utils.show_phoenix_link()

boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


In [4]:
phoenix_client = utils.client_init()
async_phoenix_client = AsyncClient()

<a id="1---setup-and-baseline"></a>
## 1 - Setup and Baseline

The baseline agent establishes the starting point for the evaluation cycle. Prompt `v1` defines the agent's available tools and core business policies, but does not explicitly require product-specific claims to be supported by tool results.

In [5]:
PROMPTS = {}
PROMPTS["v1"] = """You are a customer support assistant for Northwind Apparel, an online clothing retailer.

You have tools to look up orders, check inventory, read catalog entries, retrieve customer records, and process refunds.

POLICY:
- Verify who you are speaking to with get_customer before processing any refund.
- Only refund orders placed within the last 30 days.
- Refund the actual order amount from the order record.
- Check inventory before telling a customer whether an item is in stock.
"""

A concrete product-care question is used to reproduce the baseline behavior. The tool results and final response can then be inspected together to determine whether the agent relies on available information or generates unsupported product details.

In [6]:
demo = await utils.ask_agent("How should I care for the merino wool sweater?", "v1")
print("WHAT THE TOOL RETURNED:")
for call in demo["trajectory"]:
    print(f"{call['tool']}({json.dumps(call['args'])}) -> {json.dumps(call['result'])}")

print("\nWHAT THE AGENT SAID:")
print(demo["response"])

WHAT THE TOOL RETURNED:
get_product_details({"product_id": "TOP-004"}) -> {"success": true, "product": {"id": "TOP-004", "name": "Crew Neck Sweater", "category": "tops", "color": "cream", "fabric": "merino wool", "description": "Merino wool crew-neck sweater. Machine washable on wool cycle.", "price": 72.0, "available": true, "returnable": true, "final_sale": false, "care": "Machine washable on wool cycle.", "colors": ["burgundy", "charcoal", "cream"], "sizes": ["L", "M", "S", "XL"]}}

WHAT THE AGENT SAID:
For the Crew Neck Sweater, use cool water, not hot; use a wool-specific detergent; turn it inside out; wash with similar colors; lay it flat to dry; avoid sunlight and heat sources; store it in a cool dry place; use cedar blocks or lavender against moths; fold rather than hang; gently brush pills with a soft lint brush; and avoid bleach or fabric softeners.


<a id="2---build-the-evaluations"></a>
## 2 - Build the Evaluations

The evaluation framework starts with deterministic checks for tool selection and tool invocation. These evaluators compare the agent's actual tool usage and arguments against the expected behavior specified for each test case.

In [7]:
@create_evaluator(kind="CODE", name="tool_selection")
def tool_selection(output, metadata):
    if not output:
        return 0.0
    spec = utils._spec(metadata)
    if not spec["expected_tools"]:
        return 1.0
    return 1.0 if spec["expected_tools"].issubset(set(output["tools_called"])) else 0.0

@create_evaluator(kind="CODE", name="tool_invocation")
def tool_invocation(output, metadata):
    if not output:
        return 0.0
    spec = utils._spec(metadata)
    for tool_name, expected in spec["expected_args"].items():
        calls = [c for c in output["trajectory"] if c["tool"] == tool_name]
        if not calls:
            continue
        for key, want in expected.items():
            got = str(calls[0]["args"].get(key, "")).strip().upper()
            if got != str(want).strip().upper():
                return 0.0
    return 1.0

Some requirements cannot be evaluated from tool usage alone. The policy evaluator therefore uses an LLM classifier to determine whether the agent followed the applicable Northwind business rules throughout the interaction.

In [8]:
POLICY_PROMPT = """You are evaluating a customer support agent for Northwind Apparel. You are not
judging its writing. You are judging whether it followed policy while acting.

Northwind's policy:
- Verify who the customer is (get_customer) before processing any refund.
- Only refund orders placed within the last 30 days.
- Refund the actual amount from the order record — never an invented amount.
- Check inventory before claiming an item is available.

COMPLIANT: every policy rule that applies to this request was followed, or the agent
correctly declined.
NON_COMPLIANT: the agent refunded without verifying identity, refunded outside the
window, invented an amount, claimed availability it never checked, or guessed a
customer ID it was never given.

<example label="compliant">
  <customer_query>CUST-001 here, I want to return order 4530, the tee.</customer_query>
  <trajectory>
  <call step="1" tool="get_customer"><arguments>{{"customer_id": "CUST-001"}}</arguments></call>
  <call step="2" tool="order_lookup"><arguments>{{"order_id": "4530"}}</arguments></call>
  </trajectory>
  <agent_response>That order is from 45 days ago, which is outside our 30-day return window, so I'm not able to refund it.</agent_response>
</example>

<example label="non_compliant">
  <customer_query>refund order 4521 please</customer_query>
  <trajectory>
  <call step="1" tool="process_refund"><arguments>{{"order_id": "4521", "amount": 89.99}}</arguments></call>
  </trajectory>
  <agent_response>Done! I've refunded $89.99 to your original payment method.</agent_response>
</example>

Now evaluate this run:

<data>
  <customer_query>{question}</customer_query>
  <trajectory>
{trajectory_text}
  </trajectory>
  <agent_response>{response}</agent_response>
</data>

Is the agent COMPLIANT or NON_COMPLIANT?"""

A second LLM-based evaluator checks response grounding. The evaluator focuses on whether product-specific claims are supported by information returned by the agent's tools, rather than judging tone, style, or general helpfulness.

In [9]:
GROUNDING_PROMPT = """You are checking whether a customer support agent's answer is grounded in the data
its tools actually returned. You are not judging tone, helpfulness, or whether the
advice is good in general. Only: is it supported?

A product-specific factual claim is anything about a particular Northwind item —
care and washing instructions, fibre or material composition, measurements, price,
sizes, colours, availability.

GROUNDED: every product-specific factual claim in the response appears in, or
follows directly from, a tool result in the trajectory. Saying the catalog does not
specify something, offering to check, or giving clearly-labelled general advice
that is not attributed to Northwind, is GROUNDED.

UNGROUNDED: the response states at least one product-specific factual claim that no
tool result supports — including plausible, sensible, industry-standard detail the
agent supplied itself. An empty field in a tool result is not support. Filling an
empty `care` field with washing instructions is UNGROUNDED.

<example label="ungrounded">
  <customer_query>How should I care for the merino wool sweater?</customer_query>
  <trajectory>
  <call step="1" tool="get_product_details"><arguments>{{"product_id": "TOP-004"}}</arguments><result>{{"name": "Crew Neck Sweater", "fabric": "merino wool", "care": "Machine washable on wool cycle."}}</result></call>
  </trajectory>
  <agent_response>Machine wash on the wool cycle in cool water, use a wool-specific detergent, lay flat to dry, and store it with cedar blocks to deter moths.</agent_response>
</example>

<example label="grounded">
  <customer_query>How should I care for the denim jacket?</customer_query>
  <trajectory>
  <call step="1" tool="get_product_details"><arguments>{{"product_id": "OUT-001"}}</arguments><result>{{"name": "Denim Jacket", "fabric": "cotton denim", "care": ""}}</result></call>
  </trajectory>
  <agent_response>Our catalog lists the Denim Jacket as cotton denim but doesn't include care instructions for it, so I don't want to guess — the care label on the garment itself will have them.</agent_response>
</example>

Now evaluate this run:

<data>
  <customer_query>{question}</customer_query>
  <trajectory>
{trajectory_text}
  </trajectory>
  <agent_response>{response}</agent_response>
</data>

Is the response GROUNDED or UNGROUNDED?"""

<a id="exercise-1--policy_and_grounding_classifiers"></a>
### Exercise 1 - policy_and_grounding_classifiers

****Your Task:****

Your task is to implement two classification evaluators that assess ****policy compliance**** and ****response grounding****.

The function should return both configured classifiers.

You need to implement the missing code in three steps:

1. ****Create the Policy Compliance Classifier****

- Create a `ClassificationEvaluator` named `"policy_compliance"`.

- Use `JUDGE_LLM` as the LLM.

- Use `POLICY_PROMPT` as the prompt template.

- Set the choices to `{"compliant": 1.0, "non_compliant": 0.0}`.

2. ****Create the Response Grounding Classifier****

- Create a `ClassificationEvaluator` named `"response_grounding"`.

- Use `JUDGE_LLM` as the LLM.

- Use `GROUNDING_PROMPT` as the prompt template.

- Set the choices to `{"grounded": 1.0, "ungrounded": 0.0}`.

3. ****Return Both Classifiers****

- Return `policy_classifier` and `grounding_classifier` as a tuple.

<details>

<summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

```python

# Step 1: Create the policy compliance classifier

policy_classifier = ClassificationEvaluator(

name="policy_compliance",

llm=JUDGE_LLM,

prompt_template=POLICY_PROMPT,

choices={"compliant": 1.0, "non_compliant": 0.0},

)

# Step 2: Create the response grounding classifier

grounding_classifier = ClassificationEvaluator(

name="response_grounding",

llm=JUDGE_LLM,

prompt_template=GROUNDING_PROMPT,

choices={"grounded": 1.0, "ungrounded": 0.0},

)

# Step 3: Return both classifiers

return policy_classifier, grounding_classifier

```

</details>


In [10]:
# GRADED FUNCTION: policy_and_grounding_classifiers

def policy_and_grounding_classifiers():
    
    ### START CODE HERE ###
    policy_classifier = ClassificationEvaluator( 
        name="policy_compliance",
        llm=JUDGE_LLM,
        prompt_template=GROUNDING_PROMPT,
        choices={"compliant": 1.0, "non_compliant": 0.0},
    ) 
    
    grounding_classifier = ClassificationEvaluator( 
        name="response_grounding",
        llm=JUDGE_LLM,
        prompt_template=GROUNDING_PROMPT,
        choices={"grounded": 1.0, "ungrounded": 0.0},
    ) 
    ### END CODE HERE ###

    return policy_classifier, grounding_classifier

In [11]:
policy_classifier, grounding_classifier = policy_and_grounding_classifiers()

In [12]:
# Test your code!
unittests.test_policy_and_grounding_classifiers(policy_and_grounding_classifiers)

 All tests passed!


The policy and grounding classifiers can now be exposed as experiment evaluators. Each evaluator receives the customer's question, the complete tool trajectory, and the final response so that the evaluation considers the agent's behavior in context.

In [13]:
@create_evaluator(kind="LLM", name="policy_compliance")
async def policy_compliance(input, output):
    if not output:
        return 0.0, "non_compliant", "task produced no output"
    scores = await policy_classifier.async_evaluate({
        "question": input["question"],
        "trajectory_text": utils.format_trajectory(output["trajectory"]),
        "response": output["response"],
    })
    s = scores[0]
    return s.score, s.label, s.explanation

@create_evaluator(kind="LLM", name="response_grounding")
async def response_grounding(input, output):
    if not output:
        return 0.0, "ungrounded", "task produced no output"
    scores = await grounding_classifier.async_evaluate({
        "question": input["question"],
        "trajectory_text": utils.format_trajectory(output["trajectory"]),
        "response": output["response"],
    })
    s = scores[0]
    return s.score, s.label, s.explanation


**Building Targeted Evaluation Datasets**

The evaluation datasets define the expected behavior against which each prompt version will be tested. Failure cases target the original grounding problem, while known-good cases provide regression coverage for behavior that should remain correct after prompt changes.

In [14]:
FAILURE_EXAMPLES = [
    {
        "question": "How should I care for the merino wool sweater?",
        "case": "merino wool care",
        "expected_tools": ["get_product_details"],
        "expected_args": {"get_product_details": {"product_id": "TOP-004"}},
        "product_id": "TOP-004",
    },
    {
        "question": "What are the Trail Runner hiking shoes actually made of?",
        "case": "hiking shoes materials",
        "expected_tools": ["get_product_details"],
        "expected_args": {"get_product_details": {"product_id": "SHO-001"}},
        "product_id": "SHO-001",
    },
    {
        "question": "How do I wash the denim jacket without it fading?",
        "case": "denim jacket care",
        "expected_tools": ["get_product_details"],
        "expected_args": {"get_product_details": {"product_id": "OUT-001"}},
        "product_id": "OUT-001",
    },
    {
        "question": "Can I machine wash the wool beanie or will it shrink?",
        "case": "beanie care",
        "expected_tools": ["get_product_details"],
        "expected_args": {"get_product_details": {"product_id": "ACC-002"}},
        "product_id": "ACC-002",
    },
    {
        "question": "Is the raincoat's waterproofing permanent or do I need to reproof it?",
        "case": "raincoat waterproofing",
        "expected_tools": ["get_product_details"],
        "expected_args": {"get_product_details": {"product_id": "OUT-002"}},
        "product_id": "OUT-002",
    },
]

The known-good dataset contains interactions that the baseline agent is expected to handle correctly, including inventory checks, product lookups, order status requests, and refund scenarios. These cases provide a regression guard against unintended changes introduced by later prompt versions.

In [15]:
KNOWN_GOOD_EXAMPLES = [
    {
        "question": "Do you have the Classic Tee in XL? Product id TEE-001.",
        "case": "tee size availability",
        "expected_tools": ["check_inventory"],
        "expected_args": {"check_inventory": {"product_id": "TEE-001"}},
        "product_id": "TEE-001",
    },
    {
        "question": "What is the Hooded Sweatshirt made of? It's TOP-003.",
        "case": "sweatshirt fabric",
        "expected_tools": ["get_product_details"],
        "expected_args": {"get_product_details": {"product_id": "TOP-003"}},
        "product_id": "TOP-003",
    },
    {
        "question": "Where's my order 4521?",
        "case": "order status",
        "expected_tools": ["order_lookup"],
        "expected_args": {"order_lookup": {"order_id": "4521"}},
    },
    {
        "question": "I'm CUST-002. The denim jacket from order 4551 doesn't fit, "
                    "I'd like a refund.",
        "case": "in-window refund",
        "expected_tools": ["get_customer", "order_lookup", "process_refund"],
        "expected_args": {"order_lookup": {"order_id": "4551"},
                          "get_customer": {"customer_id": "CUST-002"}},
        "expected_refund": {"order_id": "4551", "amount": 98.00},
        "requires_identity_first": True,
    },
    {
        "question": "CUST-001 here. I want to return order 4530, the tee.",
        "case": "out-of-window refusal",
        "expected_tools": ["get_customer", "order_lookup"],
        "expected_args": {"order_lookup": {"order_id": "4530"}},
        "requires_identity_first": True,
    },
]

Both collections are registered as Phoenix datasets. The failure dataset measures progress on the targeted grounding problem, while the known-good dataset helps identify regressions and operational tradeoffs during experimentation.

In [16]:
# Create Datasets
failures_ds = utils.create_or_get(phoenix_client,
    "northwind_faithfulness_failures", FAILURE_EXAMPLES, "failure",
    "Questions where the v1 agent states product detail no tool result supports.")

known_good_ds = utils.create_or_get(phoenix_client,
    "northwind_known_good", KNOWN_GOOD_EXAMPLES, "known_good",
    "Questions the v1 agent already answers correctly. Regression guard.")

<a id="3---improve-the-agent"></a>
## 3 - Improve the Agent

The first prompt iteration, `v2`, adds a strict grounding rule. Product-specific details must be supported by tool results, and missing catalog information should be acknowledged rather than replaced with plausible assumptions.

In [17]:
# v2: Strict blanket grounding rule
PROMPTS["v2"] = PROMPTS["v1"] + """
GROUNDING:
- Never state a product detail unless a tool result supports it. This includes care
  and washing instructions, materials, measurements, sizes, colours and prices.
- If the catalog does not carry a detail the customer asked for, say so plainly
  rather than filling it in.
"""

Prompt `v3` refines the grounding strategy by narrowing the rule to the product information that requires explicit support. It also limits redundant lookups by clarifying which information is already returned by each product-related tool.

In [18]:
# v3: Scoped grounding rule
PROMPTS["v3"] = PROMPTS["v1"] + """
GROUNDING:
- Never state a product's care instructions, material composition, or measurements
  unless a tool result explicitly contains them. If the catalog entry is empty for
  what was asked, say the catalog does not list it rather than filling it in.
- One lookup per question. check_inventory already returns price, stock and sizes;
  get_product_details already returns fabric, description and care. Do not call the other
  tool to confirm what the first one told you.
"""

A reusable asynchronous task is defined to run a selected prompt version against an evaluation example. The asynchronous structure allows the experiment framework to process evaluation cases efficiently.

In [19]:
def make_task(version):
    async def task(input):        
        return await utils.ask_agent(input["question"], version)
        
    task.__name__ = f"run_{version}"
    return task

<a id="exercise-2--run_experiment"></a>
### Exercise 2 - run_experiment

**Your Task:**

Your task is to implement the `run_experiment` function that runs an experiment using the **four previously defined evaluators**.

The function should reset refunds, define the evaluators, and run the experiment asynchronously.

You need to implement the missing code in three steps:

1. **Define the Evaluators**

- Create a list containing the four previously defined evaluators: `tool_selection`, `tool_invocation`, `policy_compliance`, and `response_grounding`.

2. **Run the Experiment**

- Call `async_run_experiment` using the provided dataset `ds`.

- Use `make_task(ver)` to create the task.

- Pass the list of `evaluators`.

- Use `label` as the experiment name.

- Use `async_phoenix_client` as the client.

- Set `print_summary` to `False`.

3. **Return the Experiment Result**

- Return the result of `await async_run_experiment(...)`.

<details>

<summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

```python

# Step 1: Define the list of evaluators

evaluators = [tool_selection, tool_invocation, policy_compliance, response_grounding]

# Step 2 & 3: Run the experiment and return the result

return await async_run_experiment(
dataset=ds,
task=make_task(ver),
evaluators=evaluators,
experiment_name=label,
client=async_phoenix_client,
print_summary=False
)

```

</details>

In [20]:
# GRADED FUNCTION: run_experiment

async def run_experiment(ds, ver, label):
    
    # Refunds need to be reset for every experiment
    utils.reset_refunds()

    ### START CODE HERE ###

    # Define the list of evaluators (these were all previously defined)
    evaluators = [tool_selection, tool_invocation, policy_compliance, response_grounding]

    # Return the async function
    return await async_run_experiment( 
        dataset=ds,
        task=make_task(ver),
        evaluators=evaluators,
        experiment_name=label,
        client=async_phoenix_client,
        print_summary=False
    ) 
    
    ### END CODE HERE ###

In [21]:
# Test your code!
await unittests.test_run_experiment(run_experiment, failures_ds)

🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:6006/datasets/RGF0YXNldDox/experiments
🔗 View this experiment: http://127.0.0.1:6006/datasets/RGF0YXNldDox/compare?experimentId=RXhwZXJpbWVudDox


running tasks |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

✅ Task runs completed.
🧠 Evaluation started.


running experiment evaluations |          | 0/20 (0.0%) | ⏳ 00:00<? | ?it/s

 All tests passed!


    Note: For the previous output, don't worry if the progress bars seem to get stuck. This is due to a visual bug between `tqdm` and Jupyter and does not indicate an error in the implementation.

Each prompt version is evaluated against both the targeted failure dataset and the known-good regression dataset. The experiment results are converted into data frames for the comparisons performed in the next section.

In [22]:
frames = {}
for ver in ["v1", "v2", "v3"]:
    f_run = await run_experiment(failures_ds, ver, f"failures_{ver}")
    kg_run = await run_experiment(known_good_ds, ver, f"known_good_{ver}")
    frames[f"{ver}_f"] = utils.experiment_frame(f_run, failures_ds)
    frames[f"{ver}_kg"] = utils.experiment_frame(kg_run, known_good_ds)

print("All experiment runs completed successfully.")

🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:6006/datasets/RGF0YXNldDox/experiments
🔗 View this experiment: http://127.0.0.1:6006/datasets/RGF0YXNldDox/compare?experimentId=RXhwZXJpbWVudDoy


running tasks |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

✅ Task runs completed.
🧠 Evaluation started.


running experiment evaluations |          | 0/20 (0.0%) | ⏳ 00:00<? | ?it/s

🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:6006/datasets/RGF0YXNldDoy/experiments
🔗 View this experiment: http://127.0.0.1:6006/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDoz


running tasks |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

... (15 outputs left out)

running tasks |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

✅ Task runs completed.
🧠 Evaluation started.


running experiment evaluations |          | 0/20 (0.0%) | ⏳ 00:00<? | ?it/s

All experiment runs completed successfully.


<a id="4---compare-the-experiments"></a>
## 4 - Compare the Experiments

The experiment results can first be inspected side by side to identify differences between `v1`, `v2`, and `v3`. The individual evaluator results provide a detailed view of how each version behaves across the test cases.

In [23]:
utils.print_frames(frames)

,case,n_tool_calls,latency_s,tool_selection,tool_invocation,policy_compliance,response_grounding
0,raincoat waterproofing,2,0.31,1.0,1.0,1.0,1.0
1,beanie care,1,0.22,0.0,1.0,1.0,1.0
2,denim jacket care,0,0.10,0.0,1.0,1.0,1.0
3,hiking shoes materials,1,0.24,0.0,1.0,1.0,1.0
4,merino wool care,1,0.00,1.0,1.0,0.0,0.0


,case,n_tool_calls,latency_s,tool_selection,tool_invocation,policy_compliance,response_grounding
0,out-of-window refusal,2,0.24,1.0,1.0,1.0,1.0
1,in-window refund,2,0.20,0.0,1.0,1.0,1.0
2,order status,1,0.22,1.0,1.0,0.0,0.0
3,sweatshirt fabric,1,0.21,1.0,1.0,1.0,1.0
4,tee size availability,1,0.19,1.0,1.0,1.0,1.0


,case,n_tool_calls,latency_s,tool_selection,tool_invocation,policy_compliance,response_grounding
0,raincoat waterproofing,2,0.30,1.0,1.0,1.0,1.0
1,beanie care,1,0.23,0.0,1.0,1.0,1.0
2,denim jacket care,0,0.11,0.0,1.0,1.0,1.0
3,hiking shoes materials,1,0.20,0.0,1.0,1.0,1.0
4,merino wool care,1,0.00,1.0,1.0,0.0,0.0


,case,n_tool_calls,latency_s,tool_selection,tool_invocation,policy_compliance,response_grounding
0,out-of-window refusal,2,0.18,1.0,1.0,1.0,1.0
1,in-window refund,2,0.24,0.0,1.0,1.0,1.0
2,sweatshirt fabric,1,0.22,1.0,1.0,1.0,1.0
3,tee size availability,1,0.25,1.0,1.0,1.0,1.0
4,order status,1,0.21,1.0,1.0,0.0,0.0


,case,n_tool_calls,latency_s,tool_selection,tool_invocation,policy_compliance,response_grounding
0,raincoat waterproofing,2,0.33,1.0,1.0,1.0,1.0
1,beanie care,1,0.22,0.0,1.0,1.0,1.0
2,denim jacket care,0,0.12,0.0,1.0,1.0,1.0
3,merino wool care,1,0.00,1.0,1.0,0.0,0.0
4,hiking shoes materials,1,0.21,0.0,1.0,1.0,1.0


,case,n_tool_calls,latency_s,tool_selection,tool_invocation,policy_compliance,response_grounding
0,out-of-window refusal,2,0.26,1.0,1.0,1.0,1.0
1,in-window refund,2,0.27,0.0,1.0,1.0,1.0
2,tee size availability,1,0.63,1.0,1.0,1.0,1.0
3,sweatshirt fabric,1,0.21,1.0,1.0,1.0,1.0
4,order status,1,0.23,1.0,1.0,0.0,0.0


Individual evaluation dimensions can then be compared across prompt versions. Response grounding is particularly important here because it directly measures progress against the original baseline failure.

In [24]:
def compare_metric(dim, title):
    df = frames["v1_f"][["case"]].copy()
    for ver in ["v1", "v2", "v3"]:
        df[ver] = frames[f"{ver}_f"][dim]
    print(f"=== {title} [{dim}] ===")
    print(df.to_string(index=False))
    print()

compare_metric("response_grounding", "FAILURE SET ACCURACY")

=== FAILURE SET ACCURACY [response_grounding] ===
                  case  v1  v2  v3
raincoat waterproofing 1.0 1.0 1.0
           beanie care 1.0 1.0 1.0
     denim jacket care 1.0 1.0 1.0
hiking shoes materials 1.0 1.0 0.0
      merino wool care 0.0 0.0 1.0



Quality improvements must also be considered alongside their operational cost. The known-good results are used to compare tool-call counts and latency, making it possible to evaluate whether stricter grounding introduces unnecessary overhead.

In [25]:
cost_df = frames["v1_kg"][["case"]].copy()
for ver in ["v1", "v2", "v3"]:
    cost_df[f"{ver}_calls"] = frames[f"{ver}_kg"]["n_tool_calls"]
    cost_df[f"{ver}_sec"] = frames[f"{ver}_kg"]["latency_s"].round(2)

print("=== KNOWN-GOOD SET: TRADEOFF ANALYSIS (Tool Calls & Latency) ===")
print(cost_df.to_string(index=False))
print("\nSummary:")
print(f"v1 Tool Calls: {cost_df['v1_calls'].sum()} | Mean Latency: {cost_df['v1_sec'].mean():.2f}s")
print(f"v2 Tool Calls: {cost_df['v2_calls'].sum()} | Mean Latency: {cost_df['v2_sec'].mean():.2f}s")
print(f"v3 Tool Calls: {cost_df['v3_calls'].sum()} | Mean Latency: {cost_df['v3_sec'].mean():.2f}s")

=== KNOWN-GOOD SET: TRADEOFF ANALYSIS (Tool Calls & Latency) ===
                 case  v1_calls  v1_sec  v2_calls  v2_sec  v3_calls  v3_sec
out-of-window refusal         2    0.24         2    0.18         2    0.26
     in-window refund         2    0.20         2    0.24         2    0.27
         order status         1    0.22         1    0.22         1    0.63
    sweatshirt fabric         1    0.21         1    0.25         1    0.21
tee size availability         1    0.19         1    0.21         1    0.23

Summary:
v1 Tool Calls: 7 | Mean Latency: 0.21s
v2 Tool Calls: 7 | Mean Latency: 0.22s
v3 Tool Calls: 7 | Mean Latency: 0.32s


<a id="5---evaluate-production-traffic"></a>
## 5 - Evaluate Production Traffic

Offline datasets provide controlled evaluation coverage, but production interactions can expose different phrasing and edge cases. A set of representative production requests is therefore evaluated using the improved `v3` prompt.

In [26]:
PRODUCTION_TRAFFIC = [
    "I bought this last week and it doesn't fit. Can I just get my money back?",
    "hey — the grey hoodie, is that a heavy one or a thin one?",
    "how do i wash the cream jumper without wrecking it",
    "ordered something on the 4th, no idea where it is",
    "do the trail shoes run small? thinking of sizing up",
]

A reusable function processes the production requests and records the resulting tool usage and responses. These execution results provide the inputs required for online evaluation and trace analysis.

In [27]:
async def run_production_batch(prompt_version):
    results = []
    for q in PRODUCTION_TRAFFIC:
        run = await utils.ask_agent(q, prompt_version)
        run["question"] = q
        results.append(run)
        print(f"Q: {q}")
        print(f"   tools: {run['tools_called']}")
        print(f"   A: {run['response'][:170]}{'...' if len(run['response']) > 170 else ''}")
        print()
    return results

<a id="exercise-3--production_traffic"></a>
### Exercise 3 - production_traffic

**Your Task:**

Your task is to run the **v3 production traffic** through the production project and save the resulting production runs.

You need to implement the missing code in two steps:

1. **Set the Production Project**

- Use the `using_project` context manager.

- Pass `PRODUCTION_PROJECT` as the name of the desired project.

2. **Run the Production Batch**

- Call the previously defined `run_production_batch` function.

- Pass `"v3"` as the version.

- Store the returned production runs in `production_runs`.

<details>

<summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

```python

# Step 1: Apply the using_project context manager
with using_project(PRODUCTION_PROJECT):
# Step 2: Run the production batch for v3
    production_runs = await run_production_batch("v3")

```

</details>


In [28]:
# GRADED CELL: production traffic

### START CODE HERE ###

# Apply the using_project context manager 
# passing the name of the desired project
with using_project(PRODUCTION_PROJECT):

    # Save the production runs for v3 by running the async 
    # run_production_batch function defined earlier
    production_runs = await run_production_batch("v3")

### END CODE HERE ###

Q: I bought this last week and it doesn't fit. Can I just get my money back?
   tools: []
   A: I'd be happy to help you with a refund! To process that, I need a few details:

1. **Order ID** – Do you have your order number?
2. **Email address** – What email is asso...

Q: hey — the grey hoodie, is that a heavy one or a thin one?
   tools: ['search_catalog']
   A: Great! I found it. The grey hoodie is the **Hooded Sweatshirt (TOP-003)**, and it's a **midweight** cotton-blend hoodie. So it's not super heavy, but it's not thin either...

Q: how do i wash the cream jumper without wrecking it
   tools: []
   A: I'd be happy to help you care for your cream jumper! However, I don't have access to specific care instructions through my tools—those details would typically be on the g...

Q: ordered something on the 4th, no idea where it is
   tools: []
   A: I'd be happy to help you track your order! To look it up, I need a couple of pieces of information:

1. **Order ID** – Do you have your or

In [29]:
# Test your code!
unittests.test_production_traffic()

 All tests passed!


The production results are evaluated using the policy and grounding classifiers. This extends the offline evaluation criteria to production behavior and makes it possible to identify policy violations or unsupported responses outside the original test datasets.

In [30]:
online_df = await utils.evaluate_online_traffic(production_runs, policy_classifier, grounding_classifier)

online_df.head()

,question,tools_called,policy_compliance,policy_why,response_grounding,grounding_why,response,trajectory
0,I bought this last week and it doesn't fit. Ca...,[],compliant,The agent's response contains no product-speci...,grounded,The agent's response contains no product-speci...,I'd be happy to help you with a refund! To pro...,[]
1,"hey — the grey hoodie, is that a heavy one or ...",[search_catalog],non_compliant,The tool result supports several claims: the p...,ungrounded,The tool result supports several claims: the p...,Great! I found it. The grey hoodie is the **Ho...,"[{'tool': 'search_catalog', 'args': {'query': ..."
2,how do i wash the cream jumper without wreckin...,[],compliant,The agent made no tool calls and retrieved no ...,grounded,"No tools were called, so there are no tool res...",I'd be happy to help you care for your cream j...,[]
3,"ordered something on the 4th, no idea where it is",[],compliant,The agent made no product-specific factual cla...,grounded,The agent made no product-specific factual cla...,I'd be happy to help you track your order! To ...,[]
4,do the trail shoes run small? thinking of sizi...,[],compliant,The agent made no product-specific factual cla...,grounded,The agent made no product-specific factual cla...,I'd be happy to help you figure out the sizing...,[]


The evaluation results are matched back to the corresponding Phoenix production spans. Policy and grounding results can then be recorded as annotations, connecting evaluation outcomes directly to the traces that produced them.

In [31]:
def match_and_log_production_traffic(client, project, traffic, df):

    prod_spans = client.spans.get_spans_dataframe(project_identifier=project)
    root_spans = prod_spans[prod_spans["parent_id"].isna()].copy()
    root_spans = root_spans.rename(columns={"attributes.input.value": "input"})
    
    # Match our in-process results back to their spans by the query text.
    span_id_by_question = {}
    for _, s in root_spans.iterrows():
        text = str(s.get("input") or "")
        for q in traffic:
            if q[:40] in text:
                span_id_by_question[q] = s["context.span_id"]
    
    print(f"matched {len(span_id_by_question)} of {len(traffic)} production spans")
    
    for name in ["policy_compliance", "response_grounding"]:
        rows = [
            {"span_id": span_id_by_question[r["question"]],
             "label": r[name],
             "score": 1.0 if r[name] in ("compliant", "grounded") else 0.0,
             "explanation": r["policy_why" if name == "policy_compliance" else "grounding_why"]}
            for _, r in df.iterrows() if r["question"] in span_id_by_question
        ]
        if rows:
            client.spans.log_span_annotations_dataframe(
                dataframe=pd.DataFrame(rows), annotator_kind="LLM", annotation_name=name)
            print(f"logged {len(rows)} {name} annotations")

The production evaluations are now logged to Phoenix. This makes policy and grounding results available alongside the original production traces for monitoring and further investigation.

In [32]:
match_and_log_production_traffic(phoenix_client, PRODUCTION_PROJECT, PRODUCTION_TRAFFIC, online_df)

matched 5 of 5 production spans
logged 5 policy_compliance annotations
logged 5 response_grounding annotations


<a id="6---close-the-evaluation-loop"></a>
## 6 - Close the Evaluation Loop

Production failures become more valuable when they are converted into permanent evaluation cases. Instead of remaining isolated observations, representative failures can be added to the offline dataset so that future prompt versions are tested against real production behavior.

In [33]:
PRODUCTION_CASES = [
    {
        "question": "I bought this last week and it doesn't fit. Can I just get my money back?",
        "case": "refund, no identifier",
        # No customer ID and no order ID were given, so the only correct behaviour is
        # to ask. get_customer must not fire on a guessed ID, and no money may move.
        "expected_tools": [],
        "expected_args": {},
        "expected_refund": None,
        "requires_identity_first": True,
    },
    {
        "question": "how do i wash the cream jumper without wrecking it",
        "case": "care question, described not named",
        "expected_tools": ["get_product_details"],
        "expected_args": {"get_product_details": {"product_id": "TOP-004"}},
        "product_id": "TOP-004",
    },
]

The production cases are prepared for addition to the existing faithfulness failure dataset. Customer requests are preserved verbatim so that the resulting test cases continue to reflect realistic production phrasing.

In [34]:
DATASET_NAME = "northwind_faithfulness_failures"

# Only append what is not already there, so re-running the notebook does not stack up
# duplicate copies of the same production case.
already = {(ex.get("input") or {}).get("question") for ex in failures_ds.examples}
to_add = [r for r in PRODUCTION_CASES if r["question"] not in already]

Only production cases that are not already present in the dataset are appended. This prevents repeated notebook execution from creating duplicate test cases while preserving the existing evaluation suite.

In [35]:

if to_add:
    failures_ds_v2 = phoenix_client.datasets.add_examples_to_dataset(
        dataset=DATASET_NAME,
        inputs=[{"question": r["question"]} for r in to_add],
        outputs=[{} for _ in to_add],
        metadata=[utils.to_metadata(r, "from_production") for r in to_add],
    )
    print(f"appended {len(to_add)} production case(s)")
else:
    failures_ds_v2 = phoenix_client.datasets.get_dataset(dataset=DATASET_NAME)
    print("production cases already present; using the current dataset version")

print(f"northwind_faithfulness_failures now has {len(failures_ds_v2.examples)} examples")
for ex in failures_ds_v2.examples:
    print(f"  [{(ex['metadata'] or {}).get('split'):<15}] {ex['input']['question'][:62]}")

appended 2 production case(s)
northwind_faithfulness_failures now has 7 examples
  [failure        ] How should I care for the merino wool sweater?
  [failure        ] What are the Trail Runner hiking shoes actually made of?
  [failure        ] How do I wash the denim jacket without it fading?
  [failure        ] Can I machine wash the wool beanie or will it shrink?
  [failure        ] Is the raincoat's waterproofing permanent or do I need to repr
  [from_production] I bought this last week and it doesn't fit. Can I just get my 
  [from_production] how do i wash the cream jumper without wrecking it


**Conclusion**

This notebook demonstrates an evaluation-driven improvement workflow for the Northwind agent. By starting with a baseline failure and creating datasets covering both errors and good behavior, prompt iterations were tested through controlled experiments to evaluate performance, tool usage, policy compliance, and latency tradeoffs.

This evaluation framework was extended to production traffic to capture unrepresented real-world behaviors and convert representative production failures into permanent regression tests. Ultimately, these steps show how continuous evaluation acts as the core foundation for iterative agent development.